# 🤖 AutoRig3D — UniRig AI 자동 리깅

GLB 3D 모델을 업로드하면 UniRig(SIGGRAPH 2025)가 자동으로 스켈레톤 + 스킨닝을 생성합니다.

**사용법:**
1. 런타임 → GPU로 변경 (T4 이상)
2. 셀을 순서대로 실행
3. GLB 파일 업로드
4. 리깅된 GLB 다운로드

## Step 0: GPU 확인

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB" if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "GPU가 필요합니다! 런타임 → 런타임 유형 변경 → T4 GPU 선택"

## Step 1: UniRig 설치 (최초 1회, ~3분)

In [ ]:
import os, subprocess, sys

if not os.path.exists('/content/UniRig'):
    # 1. UniRig 클론
    !git clone https://github.com/VAST-AI-Research/UniRig /content/UniRig
    %cd /content/UniRig

    # 2. bpy 제거 + 핵심 의존성 설치
    !sed -i '/^bpy/d' requirements.txt

    !pip install -q lightning pytorch_lightning transformers einops omegaconf \
        python-box addict timm fast-simplification trimesh open3d pyrender \
        huggingface_hub wandb psutil

    !pip install -q -r requirements.txt 2>&1 | tail -3

    # 3. cumm + spconv (핵심 — 순서 중요)
    import torch
    cuda_ver = torch.version.cuda
    print(f"CUDA: {cuda_ver}, PyTorch: {torch.__version__}")

    # cumm 먼저 설치 (spconv 의존성)
    !pip install -q cumm-cu120 2>/dev/null || \
     pip install -q cumm-cu121 2>/dev/null || \
     pip install -q cumm 2>/dev/null || true

    # spconv 설치
    !pip install -q spconv-cu120 2>/dev/null || \
     pip install -q spconv-cu121 2>/dev/null || \
     pip install -q spconv 2>/dev/null || true

    # 4. torch_scatter, torch_cluster
    torch_ver = torch.__version__.split('+')[0]
    cuda_short = cuda_ver.replace('.', '')[:3]
    !pip install -q torch_scatter torch_cluster \
        -f https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_short}.html 2>/dev/null || true

    # 5. flash_attn (선택적)
    !pip install -q flash_attn 2>/dev/null || echo "flash_attn 건너뜀 (선택적)"

    # 6. extract.py를 trimesh 기반으로 패치 (bpy 대체)
    patch_code = '''
import os, sys, json, argparse
import numpy as np
import trimesh

def extract_mesh_trimesh(input_path, output_folder, target_count=50000):
    """bpy 없이 trimesh로 메쉬 추출 — UniRig inference용."""
    os.makedirs(output_folder, exist_ok=True)

    # 메쉬 로드
    scene = trimesh.load(input_path, force='scene')
    if isinstance(scene, trimesh.Scene):
        meshes = [g for g in scene.geometry.values() if isinstance(g, trimesh.Trimesh)]
        if not meshes:
            raise ValueError(f"메쉬를 찾을 수 없습니다: {input_path}")
        mesh = trimesh.util.concatenate(meshes)
    else:
        mesh = scene

    # 면 수 줄이기
    if len(mesh.faces) > target_count:
        mesh = mesh.simplify_quadric_decimation(target_count)

    vertices = np.array(mesh.vertices, dtype=np.float32)
    faces = np.array(mesh.faces, dtype=np.int64)
    vertex_normals = np.array(mesh.vertex_normals, dtype=np.float32)
    face_normals = np.array(mesh.face_normals, dtype=np.float32)

    # raw_data.npz 저장 (UniRig이 기대하는 형식)
    basename = os.path.splitext(os.path.basename(input_path))[0]
    save_dir = os.path.join(output_folder, basename)
    os.makedirs(save_dir, exist_ok=True)

    np.savez(
        os.path.join(save_dir, "raw_data.npz"),
        vertices=vertices,
        faces=faces,
        vertex_normals=vertex_normals,
        face_normals=face_normals,
        joints=np.zeros((0, 3), dtype=np.float32),
        parents=np.zeros((0,), dtype=np.int64),
        bone_names=np.array([], dtype=str),
        skin_weights=np.zeros((len(vertices), 0), dtype=np.float32),
        local_transforms=np.zeros((0, 4, 4), dtype=np.float32),
    )
    print(f"추출 완료: {save_dir}/raw_data.npz (vertices: {len(vertices)}, faces: {len(faces)})")
    return save_dir

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--target_count", type=int, default=50000)
    args = parser.parse_args()
    extract_mesh_trimesh(args.input, args.output, args.target_count)
'''
    with open('/content/UniRig/src/data/extract_trimesh.py', 'w') as f:
        f.write(patch_code)

    # extract.sh를 trimesh 버전으로 교체
    extract_sh_patch = '''#!/bin/bash
INPUT=""
OUTPUT="tmp"
TARGET_COUNT=50000
while [[ $# -gt 0 ]]; do
    case $1 in
        --input) INPUT="$2"; shift 2;;
        --output) OUTPUT="$2"; shift 2;;
        --target_count) TARGET_COUNT="$2"; shift 2;;
        *) shift;;
    esac
done
python -m src.data.extract_trimesh --input "$INPUT" --output "$OUTPUT" --target_count $TARGET_COUNT
echo "done"
'''
    with open('/content/UniRig/launch/inference/extract.sh', 'w') as f:
        f.write(extract_sh_patch)

    print("✅ extract.py → trimesh 기반으로 패치 완료")
    print("✅ UniRig 설치 완료!")
else:
    %cd /content/UniRig
    print("✅ UniRig 이미 설치됨")

# 의존성 확인
print("\n--- 의존성 확인 ---")
for pkg in ['torch', 'lightning', 'transformers', 'trimesh', 'open3d', 'spconv']:
    try:
        __import__(pkg)
        print(f"  ✅ {pkg}")
    except ImportError:
        print(f"  ❌ {pkg}")

## Step 2: GLB 파일 업로드

In [ ]:
from google.colab import files
import shutil

# 업로드
print("GLB 파일을 업로드하세요...")
uploaded = files.upload()

# 입력 파일 저장
input_filename = list(uploaded.keys())[0]
input_path = f"/content/input/{input_filename}"
os.makedirs('/content/input', exist_ok=True)
with open(input_path, 'wb') as f:
    f.write(uploaded[input_filename])

print(f"✅ 업로드 완료: {input_filename} ({len(uploaded[input_filename]) / 1024:.0f}KB)")

## Step 3: 스켈레톤 생성 (~1-2분)

In [ ]:
os.makedirs('/content/output', exist_ok=True)
skeleton_path = '/content/output/skeleton.fbx'

print("🦴 스켈레톤 생성 중...")
!bash launch/inference/generate_skeleton.sh \
    --input "{input_path}" \
    --output "{skeleton_path}"

if os.path.exists(skeleton_path):
    print(f"✅ 스켈레톤 생성 완료: {os.path.getsize(skeleton_path) / 1024:.0f}KB")
else:
    print("❌ 스켈레톤 생성 실패")

## Step 4: 스킨닝 (웨이트 페인팅, ~1-2분)

In [ ]:
skin_path = '/content/output/skin.fbx'

print("🎨 스킨닝 생성 중...")
!bash launch/inference/generate_skin.sh \
    --input "{skeleton_path}" \
    --output "{skin_path}"

if os.path.exists(skin_path):
    print(f"✅ 스킨닝 완료: {os.path.getsize(skin_path) / 1024:.0f}KB")
else:
    print("❌ 스킨닝 실패")

## Step 5: 머지 (원본 메쉬 + 리깅 결합)

In [ ]:
rigged_path = '/content/output/rigged_model.glb'

print("🔗 머지 중...")
!bash launch/inference/merge.sh \
    --source "{skin_path}" \
    --target "{input_path}" \
    --output "{rigged_path}"

if os.path.exists(rigged_path):
    print(f"✅ 리깅 완료: {os.path.getsize(rigged_path) / 1024:.0f}KB")
else:
    print("❌ 머지 실패 — 스킨 FBX를 직접 다운로드합니다")
    rigged_path = skin_path

## Step 6: 결과 다운로드

In [ ]:
from google.colab import files

if os.path.exists(rigged_path):
    print(f"📥 다운로드: {rigged_path}")
    files.download(rigged_path)
else:
    print("❌ 리깅된 파일이 없습니다")

## (선택) Google Drive에 저장

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/AutoRig3D'
os.makedirs(save_dir, exist_ok=True)

import shutil
output_name = input_filename.replace('.glb', '_rigged.glb')
shutil.copy2(rigged_path, f"{save_dir}/{output_name}")
print(f"✅ Google Drive 저장: {save_dir}/{output_name}")